# Buscador de servicio y recepción

Simula un servicio pendular invertido y busca una devolución en los puntos 2, 3 o 4. Los vectores de efecto se expresan en revoluciones por segundo sobre los ejes globales. La búsqueda solo se ejecuta al pulsar el botón. Cuando el intercambio es válido, **Generar y guardar video** crea un MP4 bajo `outputs/notebooks/04_serve_return_search/`.

In [ ]:
import importlib.util
import sys
from pathlib import Path

_root = next((path for path in (Path.cwd(), *Path.cwd().parents) if (path / 'pyproject.toml').exists()), None)
_expected_python = _root / '.venv' / 'Scripts' / 'python.exe' if _root else None
if _root is None or not _expected_python.exists() or Path(sys.executable).resolve() != _expected_python.resolve() or importlib.util.find_spec('table_tennis') is None:
    raise RuntimeError("Kernel incorrecto. Ejecuta scripts/setup_environment.ps1 y selecciona 'Python (TableTennis)'.")

import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import HTML, Video, display

from table_tennis.constants import TABLE_LENGTH, TABLE_WIDTH
from table_tennis.exchange import (
    RETURN_DEPTH_X, RETURN_DIRECTION_Y, SERVICE_DEPTH_X, SERVICE_DIRECTION_Y,
    ContactSelection, RubberProperties, ServiceTargets, StrokeTargets,
)
from table_tennis.physics import apply_racket_impact, simulate_racket_impact
from table_tennis.presets.returns import PILOT_SERVICE_PARAMS
from table_tennis.search.returns import ReturnSearchConfig, search_return, search_service
from table_tennis.validation import validate_service
from table_tennis.visualization import draw_racket, draw_table, racket_gesture_path
from table_tennis.visualization.return_videos import save_exchange_video
from table_tennis.visualization.notebook_video import notebook_video_path


In [ ]:
def spin_sliders(prefix, values):
    return [widgets.FloatSlider(description=f'{prefix} ω{axis}', value=value, min=-100, max=100, step=1)
            for axis, value in zip('xyz', values)]

service_depth = widgets.Dropdown(description='Prof. servicio', options=['short', 'two_bounce', 'long'], value='short')
service_direction = widgets.Dropdown(description='Dir. servicio', options=['forehand', 'elbow', 'backhand'], value='elbow')
service_target_x = widgets.FloatSlider(description='Servicio X', min=1370, max=2740, step=10, value=SERVICE_DEPTH_X['short'])
service_target_y = widgets.FloatSlider(description='Servicio Y', min=0, max=TABLE_WIDTH, step=10, value=SERVICE_DIRECTION_Y['elbow'])
service_spin = spin_sliders('Servicio', (0, -45, 35))

return_depth = widgets.Dropdown(description='Prof. recepción', options=['short', 'two_bounce', 'long'], value='two_bounce')
return_direction = widgets.Dropdown(description='Dir. recepción', options=['forehand', 'elbow', 'backhand'], value='elbow')
return_target_x = widgets.FloatSlider(description='Recepción X', min=0, max=1370, step=10, value=RETURN_DEPTH_X['two_bounce'])
return_target_y = widgets.FloatSlider(description='Recepción Y', min=0, max=TABLE_WIDTH, step=10, value=RETURN_DIRECTION_Y['elbow'])
return_spin = spin_sliders('Recepción', (0, -45, 0))
stroke_side = widgets.ToggleButtons(description='Golpe', options=[('Derecha', 'forehand'), ('Revés', 'backhand')], value='backhand')
contact_moment = widgets.ToggleButtons(description='Punto', options=[2, 3, 4], value=4)

service_friction = widgets.FloatSlider(description='Fricción serv.', min=0.2, max=1.5, step=.05, value=1.2)
service_restitution = widgets.FloatSlider(description='Restitución serv.', min=.4, max=1.2, step=.05, value=.8)
return_friction = widgets.FloatSlider(description='Fricción rec.', min=0.2, max=1.5, step=.05, value=1.2)
return_restitution = widgets.FloatSlider(description='Restitución rec.', min=.4, max=1.2, step=.05, value=.8)
bounce_tolerance = widgets.FloatSlider(description='Tol. bote mm', min=20, max=150, step=5, value=75)
spin_tolerance = widgets.FloatSlider(description='Tol. efecto rps', min=1, max=25, step=1, value=10)
maxiter = widgets.IntSlider(description='Iteraciones', min=10, max=250, step=10, value=180)
popsize = widgets.IntSlider(description='Población', min=4, max=24, step=1, value=18)
restarts = widgets.IntSlider(description='Reinicios', min=1, max=5, step=1, value=5)
workers = widgets.IntSlider(description='Workers', min=1, max=4, step=1, value=1)
video_duration = widgets.FloatSlider(description='Duración video', min=2, max=10, step=.5, value=5)
video_fps = widgets.IntSlider(description='FPS video', min=15, max=60, step=5, value=30)

def sync_service_targets(change=None):
    service_target_x.value = SERVICE_DEPTH_X[service_depth.value]
    service_target_y.value = SERVICE_DIRECTION_Y[service_direction.value]

def sync_return_targets(change=None):
    return_target_x.value = RETURN_DEPTH_X[return_depth.value]
    return_target_y.value = RETURN_DIRECTION_Y[return_direction.value]

service_depth.observe(sync_service_targets, names='value')
service_direction.observe(sync_service_targets, names='value')
return_depth.observe(sync_return_targets, names='value')
return_direction.observe(sync_return_targets, names='value')


In [ ]:
def report_html(title, report):
    status = 'APROBADO' if report.passed else 'NO APROBADO'
    failures = ', '.join(report.violations) if report.violations else 'ninguna'
    return HTML(f'''<h4>{title}: {status}</h4><table>
      <tr><th>Error bote</th><td>{report.bounce_error_mm:.1f} mm</td></tr>
      <tr><th>Error efecto</th><td>{report.spin_error_rps:.2f} rps</td></tr>
      <tr><th>Holgura red</th><td>{report.net_clearance_mm:.1f} mm</td></tr>
      <tr><th>Incumplimientos</th><td>{failures}</td></tr></table>''')

def plot_exchange(service_result, contact_index, return_result, service_params, return_params):
    fig = plt.figure(figsize=(11, 7))
    ax = fig.add_subplot(111, projection='3d')
    draw_table(ax)
    ax.plot(*service_result.x[:, :contact_index + 1], color='tab:orange', label='servicio')
    ax.plot(*return_result.x, color='tab:purple', label='recepción')
    ax.scatter(*service_result.x[:, contact_index], color='black', s=45, label='contacto')
    for params, color in ((service_params, 'tab:red'), (return_params, 'tab:green')):
        path = racket_gesture_path(params.ball_position, params.racket_velocity)
        ax.plot(*path.T, color=color, linestyle=':', linewidth=2)
        draw_racket(ax, params.ball_position, params.racket_angle)
    ax.set_xlim(-500, TABLE_LENGTH + 500); ax.set_ylim(-400, TABLE_WIDTH + 400); ax.set_zlim(0, 1900)
    ax.set_xlabel('X (mm)'); ax.set_ylabel('Y (mm)'); ax.set_zlabel('Z (mm)')
    ax.view_init(23.5, -45); ax.legend(); plt.show()


In [ ]:
run_button = widgets.Button(description='Buscar servicio y recepción', button_style='success', icon='search')
video_button = widgets.Button(description='Generar y guardar video', button_style='info', icon='video', disabled=True)
output = widgets.Output()
video_output = widgets.Output()
last_exchange = {}

def run_search(_):
    video_button.disabled = True
    last_exchange.clear()
    video_output.clear_output()
    with output:
        output.clear_output(wait=True)
        st = ServiceTargets(
            depth=service_depth.value, direction=service_direction.value,
            spin_rps=tuple(slider.value for slider in service_spin),
            bounce_tolerance_mm=bounce_tolerance.value, spin_tolerance_rps=spin_tolerance.value,
            target_x=service_target_x.value, target_y=service_target_y.value,
        )
        rt = StrokeTargets(
            depth=return_depth.value, direction=return_direction.value,
            spin_rps=tuple(slider.value for slider in return_spin), stroke_side=stroke_side.value,
            bounce_tolerance_mm=bounce_tolerance.value, spin_tolerance_rps=spin_tolerance.value,
            target_x=return_target_x.value, target_y=return_target_y.value,
        )
        config = ReturnSearchConfig(maxiter=maxiter.value, popsize=popsize.value, restarts=restarts.value, workers=workers.value)
        default_service = (
            st.depth == 'short' and st.direction == 'elbow' and st.spin_rps == (0, -45, 35)
            and st.target_point == (SERVICE_DEPTH_X['short'], SERVICE_DIRECTION_Y['elbow'])
            and service_friction.value == 1.2 and service_restitution.value == .8
        )
        if default_service:
            service_params = PILOT_SERVICE_PARAMS
            service_result = simulate_racket_impact(service_params, t_max=3.0)
            service_report = validate_service(service_params, service_result, st)
        else:
            print('Buscando servicio…')
            service_search = search_service(st, RubberProperties(service_friction.value, service_restitution.value), config)
            service_params, service_result, service_report = service_search.params, service_search.trajectory, service_search.validation
        display(report_html('Servicio', service_report))
        if not service_report.passed:
            print('No se busca la devolución hasta obtener un servicio válido.')
            return
        print('Buscando devolución…')
        result = search_return(
            service_result, rt, ContactSelection(moment=contact_moment.value),
            RubberProperties(return_friction.value, return_restitution.value), config,
        )
        display(report_html('Devolución', result.validation))
        print('Ángulo de raqueta:', np.round(result.params.racket_angle, 2))
        print('Velocidad de raqueta (mm/s):', np.round(result.params.racket_velocity, 2))
        print('Efecto post-impacto (rps):', np.round(np.asarray(apply_racket_impact(result.params).omega) / (2*np.pi), 2))
        plot_exchange(service_result, result.contact_index, result.trajectory, service_params, result.params)
        last_exchange.update(service_params=service_params, service_result=service_result, return_search=result, service_targets=st, return_targets=rt)
        video_button.disabled = not result.validation.passed

def generate_video(_):
    if not last_exchange:
        return
    result = last_exchange['return_search']
    filename = (
        f"service-{last_exchange['service_targets'].depth}-{last_exchange['service_targets'].direction}"
        f"_return-{last_exchange['return_targets'].depth}-{last_exchange['return_targets'].direction}"
        f"-{last_exchange['return_targets'].stroke_side}-point-{result.contact.moment}.mp4"
    )
    path = notebook_video_path('04_serve_return_search', filename.removesuffix('.mp4'))
    video_button.disabled = True
    run_button.disabled = True
    with video_output:
        video_output.clear_output(wait=True)
        print(f'Generando {path}…')
        try:
            save_exchange_video(
                None,
                last_exchange['return_targets'].stroke_side,
                path,
                fps=video_fps.value,
                max_duration=video_duration.value,
                service_params=last_exchange['service_params'],
                service_result=last_exchange['service_result'],
                contact=result.contact,
                contact_index=result.contact_index,
                return_params=result.params,
                service_targets=last_exchange['service_targets'],
                return_targets=last_exchange['return_targets'],
                video_title=f"{last_exchange['service_targets'].depth} → {last_exchange['return_targets'].depth}",
            )
            print(f'Video guardado en: {path.resolve()}')
            display(Video(filename=str(path), embed=True))
        except Exception as exc:
            print(f'No se pudo generar el video: {exc}')
        finally:
            run_button.disabled = False
            video_button.disabled = not bool(last_exchange) or not result.validation.passed

video_button.on_click(generate_video)

run_button.on_click(run_search)


In [ ]:
display(widgets.VBox([
    widgets.HTML('<h3>Servicio pendular invertido</h3>'),
    widgets.HBox([service_depth, service_direction]), widgets.HBox([service_target_x, service_target_y]),
    widgets.HBox(service_spin), widgets.HBox([service_friction, service_restitution]),
    widgets.HTML('<h3>Recepción</h3>'),
    widgets.HBox([return_depth, return_direction]), widgets.HBox([return_target_x, return_target_y]),
    widgets.HBox(return_spin), widgets.HBox([stroke_side, contact_moment]),
    widgets.HBox([return_friction, return_restitution]),
    widgets.HTML('<h3>Aceptación y búsqueda</h3>'),
    widgets.HBox([bounce_tolerance, spin_tolerance]), widgets.HBox([maxiter, popsize, restarts, workers]),
    widgets.HTML('<h3>Video</h3>'), widgets.HBox([video_duration, video_fps]),
    widgets.HBox([run_button, video_button]), output, video_output,
]))
